# SOCRATES Training on Google Colab

Train a Socratic teaching agent on the SOCRATES environment.

**Requirements**:
- Runtime: GPU (T4)
- Time: ~30-45 minutes
- HuggingFace account with write token

**What this does**:
1. Installs dependencies
2. Clones SOCRATES from GitHub
3. Starts environment server
4. Runs training
5. Uploads trained model to HuggingFace Hub

## Step 1: Check GPU

In [ ]:
!nvidia-smi

## Step 2: Install Dependencies

In [ ]:
%%capture
!pip install transformers accelerate bitsandbytes peft
!pip install fastapi uvicorn websockets pydantic sentence-transformers scikit-learn
!pip install nest-asyncio

## Step 3: Clone SOCRATES Repository

In [ ]:
import os
import sys

# Go to safe directory
os.chdir('/content')

# Remove old if exists
!rm -rf /content/socrates-env

# Clone repository
print("Cloning SOCRATES repository...")
!git clone https://github.com/Shivam250124/socrates-env.git /content/socrates-env

# Setup paths
os.chdir('/content/socrates-env/socrates_env')
sys.path.insert(0, '/content/socrates-env/socrates_env')

print(f"✓ Repository cloned")
print(f"✓ Working directory: {os.getcwd()}")
print(f"✓ Python path configured")

# Verify key files
print("\nVerifying files...")
if os.path.exists('client.py'):
    print("✓ client.py found")
if os.path.exists('models.py'):
    print("✓ models.py found")
if os.path.exists('server/environment.py'):
    print("✓ server/environment.py found")
if os.path.exists('training/train_simple.py'):
    print("✓ training/train_simple.py found")

# Verify LoRA fix is present
with open('training/train_simple.py', 'r') as f:
    content = f.read()
    if 'load_in_8bit' in content and 'LoraConfig' in content:
        print("✓ 8-bit + LoRA fix confirmed")
    else:
        print("⚠ Warning: LoRA fix not found")
    
print("\n✓ Setup complete!")

## Step 4: Start Environment Server

In [ ]:
import subprocess
import time
import requests

print("Starting SOCRATES environment server...")
server_process = subprocess.Popen(
    ["uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", "7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server
print("Waiting for server to start...")
for i in range(30):
    try:
        response = requests.get("http://localhost:7860/health")
        if response.status_code == 200:
            print("✓ Server is ready!")
            print(f"Health check: {response.json()}")
            break
    except:
        time.sleep(1)
        if i % 5 == 0:
            print(f"  Still starting... ({i}s)")
else:
    print("⚠ Server might not be ready, but continuing...")

## Step 5: Test Environment Connection

In [ ]:
# Fix for Colab's event loop
import nest_asyncio
nest_asyncio.apply()

from client import SocratesEnv
from models import SocratesAction

print("Testing environment connection...")
env = SocratesEnv(base_url="ws://localhost:7860/ws")
obs = env.reset(task="foundation")
print(f"✓ Environment connected!")
print(f"\nConcept: {obs.concept_description[:80]}...")
print(f"Student says: {obs.student_response[:100]}...")

# Test step
action = SocratesAction(question="What do you think about that?")
obs, reward, done, info = env.step(action)
print(f"\n✓ Step works! Reward: {reward:.3f}")
env.close()

print("\n✓✓✓ Environment is working! ✓✓✓")

## Step 6: Configure Training

In [ ]:
# Update config for simple training
config_content = '''CONFIG = {
    "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
    "learning_rate": 2e-5,
    "batch_size": 4,
    "output_dir": "./checkpoints",
    "env_url": "ws://localhost:7860/ws",
    "use_wandb": False,
    "log_every_n_steps": 10,
    "save_every_n_episodes": 100,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "load_in_4bit": True,
}'''

with open('training/config.py', 'w') as f:
    f.write(config_content)

print("✓ Training configuration updated")
print("\nModel: Qwen2.5-1.5B-Instruct")
print("Training time: ~30-45 minutes")
print("GPU: T4 (fp16)")

## Step 7: Run Training

This takes ~30-45 minutes. The model will learn to ask Socratic questions.

In [ ]:
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60 + "\n")

!python -m training.train_simple

## Step 8: Test Trained Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Loading trained model...")
model_path = "./checkpoints/merged"  # Use merged model
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

print("\n" + "="*60)
print("TESTING TRAINED MODEL")
print("="*60 + "\n")

# Test prompt
test_prompt = """System: You are a Socratic tutor. Ask questions, never give answers.

User: I don't understand why 0.1 + 0.2 != 0.3 in Python.

Tutor:"""

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"Model response: {response}")
print("\n✓ Model is working!")

## Step 9: Upload to HuggingFace Hub

**Get your token**: https://huggingface.co/settings/tokens (needs WRITE access)

In [ ]:
from huggingface_hub import login, HfApi

print("Please enter your HuggingFace token:")
login()

print("\nUploading model to HuggingFace Hub...")
model_path = "./checkpoints/merged"  # Upload merged model
repo_id = "Shivam250124/socrates-tutor-qwen-1.5b"

api = HfApi()
api.upload_folder(
    folder_path=model_path,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload trained SOCRATES Socratic teaching model"
)

print(f"\n✓ Model uploaded successfully!")
print(f"View at: https://huggingface.co/{repo_id}")

## Step 10: Create Model Card

In [ ]:
repo_id = "Shivam250124/socrates-tutor-qwen-1.5b"

model_card = f"""---
language: en
license: mit
tags:
- socratic-teaching
- education
- reinforcement-learning
base_model: Qwen/Qwen2.5-1.5B-Instruct
---

# SOCRATES: Socratic Teaching Agent

Trained on the SOCRATES environment to teach via the Socratic method — guiding students through questions, never directly stating answers.

## Model Details

- **Base Model**: Qwen2.5-1.5B-Instruct
- **Training**: Supervised fine-tuning on Socratic dialogue examples
- **Environment**: SOCRATES
- **Training Time**: ~30-45 minutes on T4 GPU

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{repo_id}")
tokenizer = AutoTokenizer.from_pretrained("{repo_id}")

prompt = '''System: You are a Socratic tutor. Ask questions, never give answers.

User: I don't understand why 0.1 + 0.2 != 0.3 in Python.

Tutor:'''

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## Links

- **Environment**: [GitHub](https://github.com/Shivam250124/socrates-env)
"""

# Save and upload
with open(f"{model_path}/README.md", "w") as f:
    f.write(model_card)

api.upload_file(
    path_or_fileobj=f"{model_path}/README.md",
    path_in_repo="README.md",
    repo_id=repo_id,
    repo_type="model"
)

print("✓ Model card uploaded!")

## Done! 🎉

Your model is trained and uploaded!

**Model URL**: https://huggingface.co/Shivam250124/socrates-tutor-qwen-1.5b

**Next steps**:
1. Update your README with the model link
2. Test the model on HuggingFace
3. Submit your project!